# NLP Workflow - LDA Topic Modelling on full dataset using optimum configuration

## This Notebook Covers
- Retry Bag-of-words using bigrams
- Use LDA with bigrams and optimum k=50 for topic modelling
- Review topic distribution and bigrams in topics

## Libraries Used
- **Python:** pandas, tqdm, sklearn

---
## 1. Imports and Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import nltk
import re
import string
import pickle
import json
import time
from tqdm import tqdm

# NLP libraries
from nltk.corpus import stopwords
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import train_test_split

# Visualization settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

---
## 2. Load Preprocessed Data

In [2]:
print("=" * 70)
print("LOADING PREPROCESSED COMPLAINT DATA")
print("=" * 70)

# Load data
df = pd.read_csv("data/complaints_with_nlp_features_lda.csv", 
                 engine="python",
                 on_bad_lines="skip")

# Remove unnamed index if exists
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

print(f"\nLoaded {len(df):,} complaints")
print(f"   Columns: {len(df.columns)}")
print(f"   Date range: {df['Date received'].min()} to {df['Date received'].max()}")

# Show sample
df.head(3)

LOADING PREPROCESSED COMPLAINT DATA

Loaded 1,399,222 complaints
   Columns: 31
   Date range: 01/01/23 to 12/31/25


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,narrative_clean,narrative_no_stopwords,sentiment_compound,sentiment_neg,sentiment_neu,sentiment_pos,urgency_score,churn_intent,loyalty_score,text_length,word_count,narrative_cleaned,narrative_for_lda
0,01/20/25,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,I am writing to have the following information...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",CA,92345,NaN,Consent provided,Web,01/20/25,Closed with non-monetary relief,Yes,NaN,11588109,i am writing to have the following information...,writing following information removed credit f...,0.9430,0.037,0.794,0.169,0.0,0.4,0.0,662,119,writing following information removed credit f...,writing have the following information removed...
1,07/03/24,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,Reporting company used your report improperly,I am a victim of identity theft. Please delete...,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,FL,32824,NaN,Consent provided,Web,07/03/24,Closed with non-monetary relief,Yes,NaN,9416677,i am a victim of identity theft. please delete...,victim identity theft please delete remove ite...,0.2732,0.100,0.760,0.139,0.0,0.0,0.0,370,68,victim identity theft please delete remove ite...,victim identity theft. please delete remove th...
2,09/14/25,Vehicle loan or lease,Loan,Incorrect information on your report,Information belongs to someone else,"My name is XXXX XXXX, and I am formally disput...",NaN,"SANTANDER HOLDINGS USA, INC.",PA,19143,NaN,Consent provided,Web,09/14/25,Closed with explanation,Yes,NaN,15930829,"my name is xxxx xxxx, and i am formally disput...",name xxxx xxxx formally disputing fraudulent a...,-0.9042,0.151,0.766,0.083,1.0,0.4,0.0,830,140,name formally disputing fraudulent auto loan s...,"name , and formally disputing fraudulent auto ..."


---
## 3. Bag-of-Words (TF-IDF) Feature Extraction

Extract top 250 TF-IDF features after CFPB redaction cleaning.

In [3]:
print("=" * 70)
print("BAG-OF-WORDS (TF-IDF) FEATURE EXTRACTION")
print("=" * 70)

# Check cleaning effectiveness
avg_len_before = df['narrative_no_stopwords'].str.len().mean()
avg_len_after = df['narrative_clean'].str.len().mean()
reduction_pct = (avg_len_before - avg_len_after) / avg_len_before * 100

print(f"   Avg text length before: {avg_len_before:.0f} chars")
print(f"   Avg text length after:  {avg_len_after:.0f} chars")
print(f"   Reduction: {reduction_pct:.1f}%")

# Create TF-IDF vectors
print("\nCreating TF-IDF vectors (bigrams for BoW representation)...")

tfidf_vectorizer = TfidfVectorizer(
    max_features=250,
    ngram_range=(1, 2),  # Unigrams + bigrams for BoW
    min_df=20,
    max_df=0.8,
    sublinear_tf=True,
    use_idf=True
)

X_tfidf = tfidf_vectorizer.fit_transform(df['narrative_clean'])
feature_names = tfidf_vectorizer.get_feature_names_out()

print(f"   Created TF-IDF matrix: {X_tfidf.shape}")
print(f"      Documents: {X_tfidf.shape[0]:,}")
print(f"      Features: {X_tfidf.shape[1]}")

# Top features
avg_tfidf = np.asarray(X_tfidf.mean(axis=0)).flatten()
top_indices = avg_tfidf.argsort()[-20:][::-1]

print(f"\nTop 20 TF-IDF features:")
for i, idx in enumerate(top_indices, 1):
    print(f"   {i:2d}. {feature_names[idx]:30s}: {avg_tfidf[idx]:.4f}")

# Save vectorizer
with open('models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)
print("\nSaved: models/tfidf_vectorizer.pkl")

BAG-OF-WORDS (TF-IDF) FEATURE EXTRACTION
   Avg text length before: 732 chars
   Avg text length after:  1008 chars
   Reduction: -37.6%

Creating TF-IDF vectors (bigrams for BoW representation)...
   Created TF-IDF matrix: (1399222, 250)
      Documents: 1,399,222
      Features: 250

Top 20 TF-IDF features:
    1. xxxx                          : 0.1205
    2. xxxx xxxx                     : 0.0975
    3. of                            : 0.0826
    4. credit                        : 0.0765
    5. this                          : 0.0683
    6. that                          : 0.0676
    7. report                        : 0.0641
    8. on                            : 0.0621
    9. is                            : 0.0617
   10. have                          : 0.0613
   11. not                           : 0.0605
   12. in                            : 0.0591
   13. my credit                     : 0.0589
   14. information                   : 0.0554
   15. xx                            : 0.0551

---
## 4. Topic Modeling with LDA

**Objective:** Extract 50 interpretable topics using LDA  
**Method:** Use bigram with k=50

### 4.1 Perplexity Calculation - Bigram
 
**Method:** Train both models, compare test perplexity (lower = better)

In [4]:
print("=" * 70)
print("TRAIN/TEST SPLIT FOR PERPLEXITY")
print("=" * 70)

print(f"\nFull dataset: {len(df):,} complaints")
print(f"Split: 80% train, 20% test")
print(f"Random seed: 42")

train_texts, test_texts = train_test_split(
    df['narrative_for_lda'],
    test_size=0.2,
    random_state=42
)

print(f"\nSplit complete:")
print(f"   Train: {len(train_texts):,} complaints")
print(f"   Test:  {len(test_texts):,} complaints")

TRAIN/TEST SPLIT FOR PERPLEXITY

Full dataset: 1,399,222 complaints
Split: 80% train, 20% test
Random seed: 42

Split complete:
   Train: 1,119,377 complaints
   Test:  279,845 complaints


In [8]:
# ─── BIGRAM MODEL ───
print("\n" + "─" * 70)
print("BIGRAM LDA")
print("─" * 70)

vectorizer_bigram = CountVectorizer(
    max_features=8000,
    min_df=20,
    max_df=0.9,
    ngram_range=(1, 2),  # Unigrams + bigrams
    token_pattern=r'(?u)\b\w+\b',
    stop_words='english'
)

print("\nBuilding vocabulary...")
train_bow_bi = vectorizer_bigram.fit_transform(train_texts)
test_bow_bi = vectorizer_bigram.transform(test_texts)

vocab_bi = vectorizer_bigram.get_feature_names_out()
bigrams = [f for f in vocab_bi if ' ' in f]
print(f"   Vocabulary: {len(vocab_bi):,} terms ({len(bigrams):,} bigrams)")

print("\nTraining LDA...")
lda_bigram = LatentDirichletAllocation(
    n_components=50,
    max_iter=20,
    learning_method='online',
    learning_offset=50.0,
    batch_size=2048,
    evaluate_every=-1,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

start = time.time()
lda_bigram.fit(train_bow_bi)
train_time_bi = time.time() - start

# Calculate perplexity
train_perp_bi = lda_bigram.perplexity(train_bow_bi)
test_perp_bi = lda_bigram.perplexity(test_bow_bi)

print(f"\n   Training time: {train_time_bi:.1f}s")
print(f"   Train perplexity: {train_perp_bi:.2f}")
print(f"   Test perplexity:  {test_perp_bi:.2f}")


──────────────────────────────────────────────────────────────────────
BIGRAM LDA
──────────────────────────────────────────────────────────────────────

Building vocabulary...
   Vocabulary: 8,000 terms (5,045 bigrams)

Training LDA...
iteration: 1 of max_iter: 20
iteration: 2 of max_iter: 20
iteration: 3 of max_iter: 20
iteration: 4 of max_iter: 20
iteration: 5 of max_iter: 20
iteration: 6 of max_iter: 20
iteration: 7 of max_iter: 20
iteration: 8 of max_iter: 20
iteration: 9 of max_iter: 20
iteration: 10 of max_iter: 20
iteration: 11 of max_iter: 20
iteration: 12 of max_iter: 20
iteration: 13 of max_iter: 20
iteration: 14 of max_iter: 20
iteration: 15 of max_iter: 20
iteration: 16 of max_iter: 20
iteration: 17 of max_iter: 20
iteration: 18 of max_iter: 20
iteration: 19 of max_iter: 20
iteration: 20 of max_iter: 20

   Training time: 2734.8s
   Train perplexity: 597.34
   Test perplexity:  601.68


In [9]:
vocab_bi

array(['1028a', '1099c', '15usc', ..., 'zelle numerous', 'zelles', 'zero'],
      shape=(8000,), dtype=object)

In [10]:
df_bi_vocab = pd.DataFrame(vocab_bi, columns=["values"])
df_bi_vocab.to_csv("data/vocab_bi_8k.csv", index=False)

### 4.2 Extract Topics

In [11]:
print("=" * 70)
print("FINAL TOPIC EXTRACTION (FULL DATASET)")
print("=" * 70)

final_vectorizer = CountVectorizer(
    max_features=8000,
    min_df=20,
    max_df=0.9,
    ngram_range=(1, 2),  # Unigrams + bigrams
    token_pattern=r'(?u)\b\w+\b',
    stop_words='english'
)

print(f"\nProcessing {len(df):,} complaints...")
X_counts = final_vectorizer.fit_transform(df['narrative_for_lda'])
print(f"   Vocabulary size: {X_counts.shape[1]:,} terms")

# Train final LDA
print("\nTraining final LDA model")
final_lda = LatentDirichletAllocation(
    n_components=50,
    max_iter=20,
    learning_method='online',
    learning_offset=50.0,
    batch_size=2048,
    evaluate_every=-1,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

final_lda.fit(X_counts)
print("\n   LDA training complete")

FINAL TOPIC EXTRACTION (FULL DATASET)

Processing 1,399,222 complaints...
   Vocabulary size: 8,000 terms

Training final LDA model


C:\Users\shubh\hw_FAI\Python312\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


iteration: 1 of max_iter: 20
iteration: 2 of max_iter: 20
iteration: 3 of max_iter: 20
iteration: 4 of max_iter: 20
iteration: 5 of max_iter: 20
iteration: 6 of max_iter: 20
iteration: 7 of max_iter: 20
iteration: 8 of max_iter: 20
iteration: 9 of max_iter: 20
iteration: 10 of max_iter: 20
iteration: 11 of max_iter: 20
iteration: 12 of max_iter: 20
iteration: 13 of max_iter: 20
iteration: 14 of max_iter: 20
iteration: 15 of max_iter: 20
iteration: 16 of max_iter: 20
iteration: 17 of max_iter: 20
iteration: 18 of max_iter: 20
iteration: 19 of max_iter: 20
iteration: 20 of max_iter: 20

   LDA training complete


In [12]:
# Display topics
print("\n" + "=" * 70)
print("DISCOVERED TOPICS")
print("=" * 70)

feature_names = final_vectorizer.get_feature_names_out()
n_top_words = 20

topics = []
for topic_idx, topic in enumerate(final_lda.components_):
    top_indices = topic.argsort()[-n_top_words:][::-1]
    top_words = [feature_names[i] for i in top_indices]
    topics.append(top_words)
    
    print(f"\nTopic {topic_idx + 1}:")
    print(f"  {', '.join(top_words)}")

# Assign topics to documents
doc_topics = final_lda.transform(X_counts)
dominant_topics = doc_topics.argmax(axis=1)

df['dominant_topic'] = dominant_topics

print("\n" + "─" * 70)
print("Topic Distribution:")
print("─" * 70)

topic_counts = pd.Series(dominant_topics).value_counts().sort_index()
for topic_idx, count in topic_counts.items():
    pct = count / len(dominant_topics) * 100
    print(f"  Topic {topic_idx + 1}: {count:8,} ({pct:5.1f}%)")

# Save model
with open('models/lda_final_model.pkl', 'wb') as f:
    pickle.dump(final_lda, f)

with open('models/lda_final_vectorizer.pkl', 'wb') as f:
    pickle.dump(final_vectorizer, f)

print("\nSaved LDA model and vectorizer")


DISCOVERED TOPICS

Topic 1:
  information, consumer, report, inaccurate, credit, item, reasonable, disputed, reporting, reinvestigation, according, item information, accuracy, procedures, experian, promptly, reasonable procedures, unverifiable, consumer reporting, incomplete

Topic 2:
  late, payment, payments, late payment, late payments, account, credit, paid, reported, time, reporting, report, history, error, payment history, days, billing, reflect, credit report, inaccurate

Topic 3:
  usc, violation, credit, report, reporting, accounts, credit report, rights, inaccurate, according, information, accurate, violation usc, 1681i, delete, code, removed, usc 1681i, days, pursuant usc

Topic 4:
  account, bank, told, money, called, said, check, did, phone, funds, time, asked, received, days, just, sent, help, email, know, pay

Topic 5:
  balance, balance balance, owed, balance owed, items, original, original creditor, creditor, complaint, owed balance, bureaus, remove, inquired, credito

## 5. Topic Interpretation

Topic Interpretation and Grouping

In [15]:
import re
import json

print("=" * 70)
print("TOPIC INTERPRETATION (k=50, Bigrams)")
print("=" * 70)

# Get feature names
feature_names = final_vectorizer.get_feature_names_out()

def interpret_topic(top_words):
    """Broad category mapping"""
    keywords_str = ' '.join(top_words).lower()
    
    if 'identity' in keywords_str or 'theft' in keywords_str or 'fraud' in keywords_str:
        return "Identity Theft / Fraud"
    elif 'credit report' in keywords_str or 'reporting' in keywords_str:
        return "Credit Reporting Errors"
    elif 'debt' in keywords_str or 'collection' in keywords_str:
        return "Debt Collection"
    elif 'account' in keywords_str or 'payment' in keywords_str:
        return "Account / Payment Issues"
    elif 'fcra' in keywords_str or 'section' in keywords_str or 'agency' in keywords_str:
        return "Legal / FCRA Violations"
    elif 'charge' in keywords_str or 'fee' in keywords_str or 'billing' in keywords_str:
        return "Billing Disputes"
    else:
        return "General Complaints"

topic_metadata = {
    'model_config': {
        'n_topics': 50,
        'ngram_range': '(1,2)',
        'vocabulary_size': len(feature_names)
    },
    'topics': {}
}

print("\nTopic Interpretation:\n")

for topic_idx in range(50):
    # Get top 15 features
    top_indices = final_lda.components_[topic_idx].argsort()[-15:][::-1]
    top_features = [feature_names[i] for i in top_indices]
    
    # Separate bigrams, codes, unigrams
    bigrams = [f for f in top_features if ' ' in f]
    codes = [f for f in top_features if re.search(r'\d', f) and re.search(r'[a-zA-Z]', f)]
    unigrams = [f for f in top_features if ' ' not in f and not re.search(r'\d', f)]
    
    # Granular label (unique per topic)
    if len(bigrams) >= 2:
        granular_label = f"T{topic_idx:02d}_{bigrams[0].replace(' ','-')}_{bigrams[1].replace(' ','-')}"
    elif len(bigrams) == 1:
        granular_label = f"T{topic_idx:02d}_{bigrams[0].replace(' ','-')}_{unigrams[0]}"
    else:
        granular_label = f"T{topic_idx:02d}_{'_'.join(unigrams[:2])}"
    
    category_label = interpret_topic(top_features[:10])
    
    topic_metadata['topics'][topic_idx] = {
        'granular_label': granular_label,
        'category_label': category_label,
        'bigrams': bigrams[:5],
        'unigrams': unigrams[:5],
        'legal_codes': codes[:3]
    }
    
    # Display first 10 + every 5th
    if topic_idx < 10 or topic_idx % 5 == 0:
        print(f"\nTopic {topic_idx:2d}:")
        print(f"  Granular: {granular_label}")
        print(f"  Category: {category_label}")
        print(f"  Bigrams:  {', '.join(bigrams[:5])}")
        if codes:
            print(f"  Legal:    {', '.join(codes)}")

df['topic_granular'] = df['dominant_topic'].map(
    {idx: meta['granular_label'] for idx, meta in topic_metadata['topics'].items()}
)
df['topic_category'] = df['dominant_topic'].map(
    {idx: meta['category_label'] for idx, meta in topic_metadata['topics'].items()}
)

with open('outputs/topic_metadata_k50_hybrid.json', 'w') as f:
    json.dump(topic_metadata, f, indent=2)

# Show distribution by category
print(f"\nDistribution by Category (7 groups):")
print("─" * 70)

category_dist = df['topic_category'].value_counts()
for cat, count in category_dist.items():
    pct = count / len(df) * 100
    print(f"  {cat:30s}: {count:8,} ({pct:5.1f}%)")

# Show distribution by granular (top 10)
print(f"\nTop 10 Topics by Volume (Granular):")
print("─" * 70)

top_granular = df['topic_granular'].value_counts().head(10)
for label, count in top_granular.items():
    pct = count / len(df) * 100
    topic_num = int(label.split('_')[0][1:])
    category = topic_metadata['topics'][topic_num]['category_label']
    print(f"  {label:50s}: {count:6,} ({pct:4.1f}%) [{category}]")

TOPIC INTERPRETATION (k=50, Bigrams)

Topic Interpretation:


Topic  0:
  Granular: T00_item-information_information
  Category: Credit Reporting Errors
  Bigrams:  item information

Topic  1:
  Granular: T01_late-payment_late-payments
  Category: Account / Payment Issues
  Bigrams:  late payment, late payments, payment history

Topic  2:
  Granular: T02_credit-report_violation-usc
  Category: Credit Reporting Errors
  Bigrams:  credit report, violation usc
  Legal:    1681i

Topic  3:
  Granular: T03_account_bank
  Category: Account / Payment Issues
  Bigrams:  

Topic  4:
  Granular: T04_balance-balance_balance-owed
  Category: General Complaints
  Bigrams:  balance balance, balance owed, original creditor, owed balance, creditor balance

Topic  5:
  Granular: T05_credit-report_credit-reports
  Category: Credit Reporting Errors
  Bigrams:  credit report, credit reports, report noticed

Topic  6:
  Granular: T06_credit-reporting_fair-credit
  Category: Credit Reporting Errors
  Bigram

## 6. Risk Stratification

Create churn risk tiers based on churn_intent score.

In [16]:
print("=" * 70)
print("CHURN RISK STRATIFICATION")
print("=" * 70)

# Define risk tiers
def assign_risk_tier(churn_score):
    if churn_score >= 0.8:
        return 'Critical'
    elif churn_score >= 0.6:
        return 'High'
    elif churn_score >= 0.4:
        return 'Moderate'
    elif churn_score >= 0.3:
        return 'Low'
    else:
        return 'Minimal'

df['churn_risk_tier'] = df['churn_intent'].apply(assign_risk_tier)

# Display distribution
risk_dist = df['churn_risk_tier'].value_counts()

print("\nChurn Risk Distribution:")
print("─" * 70)

for tier in ['Critical', 'High', 'Moderate', 'Low', 'Minimal']:
    count = risk_dist.get(tier, 0)
    pct = count / len(df) * 100
    print(f"  {tier:10s}: {count:8,} ({pct:5.1f}%)")

print("─" * 70)
print(f"  Total:       {len(df):8,} (100.0%)")

CHURN RISK STRATIFICATION

Churn Risk Distribution:
──────────────────────────────────────────────────────────────────────
  Critical  :   19,238 (  1.4%)
  High      :   36,858 (  2.6%)
  Moderate  :  157,330 ( 11.2%)
  Low       :    8,354 (  0.6%)
  Minimal   : 1,177,442 ( 84.1%)
──────────────────────────────────────────────────────────────────────
  Total:       1,399,222 (100.0%)


---
## 7. Save final dataset for next stage

In [17]:
df.head(5)

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,narrative_clean,narrative_no_stopwords,sentiment_compound,sentiment_neg,sentiment_neu,sentiment_pos,urgency_score,churn_intent,loyalty_score,text_length,word_count,narrative_cleaned,narrative_for_lda,dominant_topic,topic_granular,topic_category,churn_risk_tier
0,01/20/25,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,I am writing to have the following information...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",CA,92345,NaN,Consent provided,Web,01/20/25,Closed with non-monetary relief,Yes,NaN,11588109,i am writing to have the following information...,writing following information removed credit f...,0.9430,0.037,0.794,0.169,0.0,0.4,0.0,662,119,writing following information removed credit f...,writing have the following information removed...,23,T23_credit-report_deleted-credit,Identity Theft / Fraud,Moderate
1,07/03/24,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,Reporting company used your report improperly,I am a victim of identity theft. Please delete...,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,FL,32824,NaN,Consent provided,Web,07/03/24,Closed with non-monetary relief,Yes,NaN,9416677,i am a victim of identity theft. please delete...,victim identity theft please delete remove ite...,0.2732,0.100,0.760,0.139,0.0,0.0,0.0,370,68,victim identity theft please delete remove ite...,victim identity theft. please delete remove th...,5,T05_credit-report_credit-reports,Credit Reporting Errors,Minimal
2,09/14/25,Vehicle loan or lease,Loan,Incorrect information on your report,Information belongs to someone else,"My name is XXXX XXXX, and I am formally disput...",NaN,"SANTANDER HOLDINGS USA, INC.",PA,19143,NaN,Consent provided,Web,09/14/25,Closed with explanation,Yes,NaN,15930829,"my name is xxxx xxxx, and i am formally disput...",name xxxx xxxx formally disputing fraudulent a...,-0.9042,0.151,0.766,0.083,1.0,0.4,0.0,830,140,name formally disputing fraudulent auto loan s...,"name , and formally disputing fraudulent auto ...",27,T27_credit-reporting_federal,Credit Reporting Errors,Moderate
3,05/01/25,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,"Upon reviewing my credit report, I have identi...",Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",TX,76105,NaN,Consent provided,Web,05/01/25,Closed with non-monetary relief,Yes,NaN,13274568,"upon reviewing my credit report, i have identi...",upon reviewing credit report identified inaccu...,0.3818,0.000,0.867,0.133,0.0,0.0,0.0,127,18,upon reviewing credit report identified inaccu...,"upon reviewing credit report, have identified ...",39,T39_credit-report_inaccurate-accounts,Credit Reporting Errors,Minimal
4,12/08/25,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account information incorrect,Everything is explained in my resolution packa...,Company believes it acted appropriately as aut...,Kubota North America Corporation,MS,391XX,NaN,Consent provided,Web,12/08/25,Closed with explanation,Yes,NaN,17837761,everything is explained in my resolution packa...,everything explained resolution package ive al...,0.3818,0.000,0.860,0.140,0.0,0.0,0.0,107,17,everything explained resolution package ive al...,everything explained resolution package ive al...,3,T03_account_bank,Account / Payment Issues,Minimal


In [18]:
print("=" * 70)
print("SAVING FINAL DATASET")
print("=" * 70)

final_columns = [
    'Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
    'Company', 'State', 'ZIP code', 'Submitted via', 
    'Company response to consumer', 'Timely response?', 'Consumer disputed?',
    'narrative_clean',
    'narrative_for_lda',
    'sentiment_compound', 'sentiment_neg', 'sentiment_neu', 'sentiment_pos',
    'urgency_score', 'churn_intent', 'loyalty_score',
    'text_length', 'word_count',
    'dominant_topic', 
    'topic_granular', 
    'topic_category', 
    'churn_risk_tier'
]

missing_cols = [col for col in final_columns if col not in df.columns]
if missing_cols:
    print(f"\nMissing columns: {missing_cols}")
    print("   These columns will be skipped in the output file.")

final_columns = [col for col in final_columns if col in df.columns]
df_final = df[final_columns].copy()

print(f"\nFinal dataset prepared:")
print(f"   Rows: {df_final.shape[0]:,}")
print(f"   Columns: {df_final.shape[1]}")
print(f"   Memory: {df_final.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

# Verify key columns
print(f"\nData completeness:")
for col in ['dominant_topic', 'topic_category', 'churn_risk_tier', 'sentiment_compound']:
    if col in df_final.columns:
        completeness = df_final[col].notna().sum() / len(df_final) * 100
        print(f"   {col:20s}: {completeness:5.1f}% complete")
    
output_path = 'data/cfpb_complaints_with_nlp_final.csv'
df_final.to_csv(output_path, index=False)

print(f"\nSaved: {output_path}")

print(f"\nSample record:")
if len(df_final) > 0:
    sample = df_final.sample(1).iloc[0]
    print(f"   Date: {sample.get('Date received', 'N/A')}")
    if 'topic_granular' in df_final.columns:
        print(f"   Topic (granular): {sample.get('topic_granular', 'N/A')}")
    if 'topic_category' in df_final.columns:
        print(f"   Topic (category): {sample.get('topic_category', 'N/A')}")
    if 'churn_risk_tier' in df_final.columns:
        print(f"   Risk tier: {sample.get('churn_risk_tier', 'N/A')}")
    if 'sentiment_compound' in df_final.columns:
        print(f"   Sentiment: {sample.get('sentiment_compound', 0):.3f}")

SAVING FINAL DATASET

Final dataset prepared:
   Rows: 1,399,222
   Columns: 27
   Memory: 3972.2 MB

Data completeness:
   dominant_topic      : 100.0% complete
   topic_category      : 100.0% complete
   churn_risk_tier     : 100.0% complete
   sentiment_compound  : 100.0% complete

Saved: data/cfpb_complaints_with_nlp_final.csv

Sample record:
   Date: 01/24/25
   Topic (granular): T09_credit-report_credit-reporting
   Topic (category): Credit Reporting Errors
   Risk tier: Minimal
   Sentiment: 0.147


In [19]:
# Save summary statistics
print("=" * 70)
print("SAVING SUMMARY STATISTICS")
print("=" * 70)

summary = {
    'dataset_info': {
        'total_complaints': len(df_final),
        'date_range': f"{df_final['Date received'].min()} to {df_final['Date received'].max()}",
        'total_features': len(final_columns),
        'timestamp': datetime.now().isoformat()
    },
    
    'nlp_features': {
        'sentiment_mean': float(df_final['sentiment_compound'].mean()),
        'sentiment_std': float(df_final['sentiment_compound'].std()),
        'urgency_mean': float(df_final['urgency_score'].mean()),
        'urgency_std': float(df_final['urgency_score'].std()),
        'churn_intent_mean': float(df_final['churn_intent'].mean()),
        'churn_intent_std': float(df_final['churn_intent'].std()),
        'loyalty_mean': float(df_final['loyalty_score'].mean()),
        'loyalty_std': float(df_final['loyalty_score'].std())
    },
    
    'risk_tier_distribution': {
        tier: int(count) 
        for tier, count in df_final['churn_risk_tier'].value_counts().items()
    },
    
    'topic_category_distribution': {},
    'topic_granular_top10': {},
    
    'lda_model_config': {
        'n_topics': 50,
        'ngram_range': '(1,2)',
        'model_type': 'bigram_lda'
    }
}

if 'topic_category' in df_final.columns:
    summary['topic_category_distribution'] = {
        category: int(count)
        for category, count in df_final['topic_category'].value_counts().items()
    }
    print("\nAdded topic category distribution (7 broad groups)")

if 'topic_granular' in df_final.columns:
    top_10_granular = df_final['topic_granular'].value_counts().head(10)
    summary['topic_granular_top10'] = {
        label: int(count)
        for label, count in top_10_granular.items()
    }
    print("Added top 10 granular topics (for detail)")

with open('outputs/nlp_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("\nSaved: outputs/nlp_summary.json")

# Display key stats
print(f"\nSummary Statistics:")
print(f"   Total complaints: {summary['dataset_info']['total_complaints']:,}")
print(f"   Date range: {summary['dataset_info']['date_range']}")
print(f"   Features: {summary['dataset_info']['total_features']}")

print(f"\n   Sentiment mean: {summary['nlp_features']['sentiment_mean']:.3f} ± {summary['nlp_features']['sentiment_std']:.3f}")
print(f"   Churn intent mean: {summary['nlp_features']['churn_intent_mean']:.3f} ± {summary['nlp_features']['churn_intent_std']:.3f}")

if summary['topic_category_distribution']:
    print(f"\n   Topic categories: {len(summary['topic_category_distribution'])}")
    for cat, count in sorted(summary['topic_category_distribution'].items(), 
                             key=lambda x: x[1], reverse=True)[:5]:
        pct = count / summary['dataset_info']['total_complaints'] * 100
        print(f"      {cat:30s}: {count:7,} ({pct:5.1f}%)")

print(f"\n   Risk tiers:")
for tier in ['Critical', 'High', 'Moderate', 'Low', 'Minimal']:
    if tier in summary['risk_tier_distribution']:
        count = summary['risk_tier_distribution'][tier]
        pct = count / summary['dataset_info']['total_complaints'] * 100
        print(f"      {tier:10s}: {count:7,} ({pct:5.1f}%)")

SAVING SUMMARY STATISTICS

Added topic category distribution (7 broad groups)
Added top 10 granular topics (for detail)

Saved: outputs/nlp_summary.json

Summary Statistics:
   Total complaints: 1,399,222
   Date range: 01/01/23 to 12/31/25
   Features: 27

   Sentiment mean: -0.050 ± 0.686
   Churn intent mean: 0.078 ± 0.194

   Topic categories: 7
      Credit Reporting Errors       : 579,533 ( 41.4%)
      Account / Payment Issues      : 334,914 ( 23.9%)
      Identity Theft / Fraud        : 192,387 ( 13.7%)
      General Complaints            : 185,317 ( 13.2%)
      Debt Collection               :  73,721 (  5.3%)

   Risk tiers:
      Critical  :  19,238 (  1.4%)
      High      :  36,858 (  2.6%)
      Moderate  : 157,330 ( 11.2%)
      Low       :   8,354 (  0.6%)
      Minimal   : 1,177,442 ( 84.1%)
